# Scanner Manufacturer Audit

**VERSION: scanner-audit-v1.0**

Checks if model performance varies by scanner manufacturer.
If GE/Siemens/Philips have very different AUCs, we're exposed to test-set mix shift.

**Runtime:** CPU-only, ~5 minutes. No GPU needed.

In [ ]:
print("VERSION: scanner-audit-v1.0")
import os, json, glob
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Find Phase 0 manifests (contain DICOM metadata)
manifest_paths = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "manifest" in f.lower() and f.endswith(".json"):
            manifest_paths.append(os.path.join(root, f))
print(f"Found {len(manifest_paths)} manifest files")
for p in manifest_paths[:5]:
    print(f"  {p}")

In [ ]:
# Extract manufacturer per study from manifests
study_manufacturer = {}
for mp in manifest_paths:
    try:
        with open(mp) as f:
            data = json.load(f)
        # Handle different manifest formats
        studies = data if isinstance(data, list) else data.get("studies", data.get("manifest", []))
        for s in studies:
            uid = s.get("StudyInstanceUID", s.get("study_uid", ""))
            mfr = s.get("Manufacturer", s.get("manufacturer", s.get("ScannerManufacturer", "UNKNOWN")))
            if uid and mfr:
                # Normalize manufacturer names
                mfr_upper = str(mfr).upper()
                if "GE" in mfr_upper:
                    mfr = "GE"
                elif "SIEMENS" in mfr_upper:
                    mfr = "SIEMENS"
                elif "PHILIPS" in mfr_upper:
                    mfr = "PHILIPS"
                elif "TOSHIBA" in mfr_upper or "CANON" in mfr_upper:
                    mfr = "TOSHIBA/CANON"
                else:
                    mfr = "OTHER"
                study_manufacturer[uid] = mfr
    except Exception as e:
        print(f"Error reading {mp}: {e}")

print(f"\nExtracted manufacturer for {len(study_manufacturer)} studies")
mfr_counts = pd.Series(list(study_manufacturer.values())).value_counts()
print(mfr_counts)

In [ ]:
# Load validation predictions from v2.3 run
# The training notebook saves predictions or we recompute from best checkpoint
import torch

# Try to find saved validation results
val_pred_path = "/kaggle/working/val_predictions.npz"
val_meta_path = "/kaggle/working/val_metadata.csv"

if os.path.exists(val_pred_path) and os.path.exists(val_meta_path):
    print("Found saved validation predictions")
    val_data = np.load(val_pred_path)
    y_true = val_data["y_true"]
    y_pred = val_data["y_pred"]
    val_meta = pd.read_csv(val_meta_path)
    val_uids = val_meta["StudyInstanceUID"].tolist()
else:
    print("No saved predictions found.")
    print("To run this audit, add prediction saving to the training notebook:")
    print("  np.savez('/kaggle/working/val_predictions.npz', y_true=y_true, y_pred=y_pred)")
    print("  val_df[['StudyInstanceUID']].to_csv('/kaggle/working/val_metadata.csv', index=False)")
    y_true = y_pred = val_uids = None

In [ ]:
if y_true is not None:
    FINDINGS = ["ACL", "MCL", "Medial_Meniscus", "Lateral_Meniscus", 
                "Medial_OA", "Lateral_OA", "PF_OA", "Effusion", 
                "Synovitis", "Bakers", "Contusion", "Fracture"]
    
    # Group by manufacturer
    results = {}
    for mfr in sorted(set(study_manufacturer.values())):
        idx = [i for i, uid in enumerate(val_uids) if study_manufacturer.get(uid) == mfr]
        if len(idx) < 10:
            print(f"{mfr}: only {len(idx)} studies, skipping")
            continue
        
        aucs = []
        for j in range(12):
            try:
                auc = roc_auc_score(y_true[idx, j], y_pred[idx, j])
                aucs.append(auc)
            except:
                aucs.append(np.nan)
        macro = np.nanmean(aucs)
        results[mfr] = {"n": len(idx), "macro_auc": macro, "per_finding": aucs}
        print(f"\n{mfr} (n={len(idx)}): macro AUC = {macro:.4f}")
        for fname, auc in zip(FINDINGS, aucs):
            print(f"  {fname}: {auc:.4f}")
    
    # Summary
    print("\n" + "="*50)
    print("SUMMARY: Per-Manufacturer Macro AUC")
    print("="*50)
    for mfr, r in sorted(results.items(), key=lambda x: x[1]["macro_auc"], reverse=True):
        print(f"  {mfr:15s}: {r['macro_auc']:.4f} (n={r['n']})")
    
    macros = [r["macro_auc"] for r in results.values()]
    spread = max(macros) - min(macros)
    print(f"\nSpread (max-min): {spread:.4f}")
    if spread > 0.05:
        print("⚠️  HIGH RISK: >0.05 spread means test-set mix shift could cost 0.03+")
    elif spread > 0.02:
        print("⚡ MODERATE: 0.02-0.05 spread, worth mitigating")
    else:
        print("✅ LOW RISK: <0.02 spread, scanner-robust")
else:
    print("Cannot run audit without validation predictions.")